[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luckeywolf/colab-torrent-to-drive/blob/main/Torrent_To_Google_Drive.ipynb)


# 🚀 Cloud Download Station (Ultimate Edition)
High-throughput torrent & magnet downloader for Google Drive with batch queuing, live public trackers, auto-archive unpacking, and real-time telemetry.

### ⚡ Feature Highlights
- **Batch Queue:** Process multiple magnet links or .torrent files sequentially in one run.
- **Auto-Archive Extraction:** Unpacks `.rar`, `.zip`, `.7z` scene releases automatically before Drive sync.
- **Live Trackers:** Injects 20+ Tier-1 public trackers dynamically for instant peer discovery.
- **Safe 2-Stage Pipeline:** Buffers locally on SSD to prevent Drive FUSE truncation, then streams sequentially.
- **Real-Time Telemetry:** Live transfer rate, elapsed time clock, ETA countdown, peer metrics, and average speed analytics.
- **Discord Webhooks:** Optional instant alerts sent to your phone/desktop when jobs finish.

In [ ]:
# ==========================================
# 🚀 Cloud Download Station (Ultimate Edition with Live Telemetry)
# ==========================================
from google.colab import drive
import os
import subprocess
import re
import shutil
import urllib.request
import json
import time
from IPython.display import HTML, display

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount("/content/drive")

# 2. Check Drive Free Space Pre-flight
try:
    total, used, free = shutil.disk_usage("/content/drive/My Drive")
    free_gb = free / (1024**3)
    total_gb = total / (1024**3)
    print(f"Google Drive Storage: {free_gb:.1f} GB free of {total_gb:.1f} GB")
except Exception:
    pass

# 3. Fast Install Dependencies
print("Initializing Aria2 & P7Zip archive unpacker...")
!apt-get install -y -qq aria2 p7zip-full > /dev/null
print("Engines ready.")

# 4. Setup paths
BASE_CACHE_DIR = "/content/download_cache"
DRIVE_SAVE_DIR = "/content/drive/My Drive/Torrent"

os.makedirs(BASE_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# 5. Fetch Live Tier-1 Public Trackers
print("Synchronizing Tier-1 BitTorrent trackers...")
trackers_arg = ""
try:
    tracker_url = "https://raw.githubusercontent.com/ngosang/trackerslist/master/trackers_best.txt"
    req = urllib.request.Request(tracker_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=3) as resp:
        trackers = [line.strip().decode("utf-8") for line in resp if line.strip()]
        if trackers:
            trackers_arg = ",".join(trackers)
            print(f"Loaded {len(trackers)} premium trackers.")
except Exception:
    print("Using DHT network.")

# 6. Multi-Target Batch Queue Setup
queue = []
print("\n" + "="*50)
print("📥 INPUT DOWNLOAD TARGETS")
print("Enter Magnet Link(s) or .torrent URL(s), one per line.")
print("Type 'done' or leave blank and press Enter when finished.")
print("(If you enter no links, you will be prompted to upload .torrent files)")
print("="*50)

while True:
    entry = input("Add Link (or 'done'): ").strip()
    if not entry or entry.lower() == "done":
        break
    queue.append({"target": entry, "name": f"Item {len(queue)+1}"})

if not queue:
    from google.colab import files
    print("\nUpload your .torrent file(s) (supports selecting multiple files):")
    uploaded = files.upload()
    for name in uploaded.keys():
        queue.append({"target": name, "name": name})

# 7. Optional Notification Webhook
webhook_url = input("\nDiscord Webhook URL (optional, press Enter to skip): ").strip()

# Helper Formatters
def format_seconds(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    elif m > 0:
        return f"{m}m {s}s"
    else:
        return f"{s}s"

def format_clock(seconds):
    seconds = int(seconds)
    m = seconds // 60
    s = seconds % 60
    return f"{m:02d}:{s:02d}"

# UI Rendering Functions
def render_progress_card(job_num, total_jobs, percent=0.0, speed="0 B", downloaded="0 B", total="0 B", elapsed_str="00:00", eta="--", peers="0", status="INITIALIZING"):
    return f"""
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #1e222b; border-radius: 8px; padding: 16px 18px; max-width: 620px; color: #d1d5db; box-shadow: 0 4px 20px rgba(0,0,0,0.5); font-size: 12px; margin: 8px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; padding-bottom: 10px; border-bottom: 1px solid #181b22; margin-bottom: 12px;">
        <div style="display: flex; align-items: center; gap: 8px; font-weight: 600; color: #f3f4f6; letter-spacing: 0.5px;">
          <span style="background: #1e2433; color: #60a5fa; padding: 2px 6px; border-radius: 4px; font-size: 10px;">QUEUE [{job_num}/{total_jobs}]</span>
          <span style="display: inline-block; width: 7px; height: 7px; border-radius: 50%; background: #10b981; box-shadow: 0 0 6px #10b981;"></span>
          <span>{status.upper()}</span>
        </div>
        <div style="color: #9ca3af; font-size: 11px; display: flex; gap: 10px;">
          <span>ELAPSED <strong style="color: #e5e7eb;">{elapsed_str}</strong></span>
          <span style="color: #374151;">|</span>
          <span>ETA <strong style="color: #38bdf8;">{eta}</strong></span>
        </div>
      </div>

      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 12px;">
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">RATE</div>
          <div style="color: #10b981; font-size: 14px; font-weight: 600; margin-top: 2px;">{speed}/s</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">TRANSFERRED</div>
          <div style="color: #e5e7eb; font-size: 13px; font-weight: 500; margin-top: 2px;">{downloaded}</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">TOTAL</div>
          <div style="color: #9ca3af; font-size: 13px; font-weight: 500; margin-top: 2px;">{total}</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">PEERS</div>
          <div style="color: #a78bfa; font-size: 14px; font-weight: 600; margin-top: 2px;">{peers}</div>
        </div>
      </div>

      <div style="background: #161922; border-radius: 3px; height: 6px; overflow: hidden; margin-bottom: 8px; border: 1px solid #232734;">
        <div style="background: #0ea5e9; height: 100%; width: {percent}%; transition: width 0.2s ease;"></div>
      </div>

      <div style="display: flex; justify-content: space-between; font-size: 11px; color: #6b7280;">
        <span>PROGRESS</span>
        <span style="color: #0ea5e9; font-weight: 600;">{percent:.2f}%</span>
      </div>
    </div>
    """

def render_completed_card(job_num, total_jobs, title_name, size_str, elapsed_str, avg_speed_str, save_path):
    return f"""
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #132e1e; border-radius: 8px; padding: 14px 18px; max-width: 620px; color: #d1d5db; font-size: 12px; margin: 8px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
        <div style="display: flex; align-items: center; gap: 8px; color: #10b981; font-weight: 600;">
          <span>✓</span>
          <span>JOB [{job_num}/{total_jobs}] COMPLETE: {size_str}</span>
        </div>
        <div style="color: #6b7280; font-size: 10px;">FLUSHED & VERIFIED</div>
      </div>
      <div style="display: flex; gap: 16px; margin: 6px 0; color: #9ca3af; font-size: 11px;">
        <span>Time Elapsed: <strong style="color: #f3f4f6;">{elapsed_str}</strong></span>
        <span>Avg Speed: <strong style="color: #34d399;">{avg_speed_str}</strong></span>
      </div>
      <div style="color: #9ca3af; font-size: 11px; font-family: monospace; word-break: break-all; background: #11141b; padding: 6px 10px; border-radius: 4px; border: 1px solid #1a202c; margin-top: 6px;">
        {save_path}
      </div>
    </div>
    """

def render_status_card(job_num, total_jobs, title, subtitle, color="#f59e0b", icon="⟳"):
    return f"""
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #222630; border-radius: 8px; padding: 14px 18px; max-width: 620px; color: #d1d5db; font-size: 12px; margin: 8px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 4px;">
        <div style="display: flex; align-items: center; gap: 8px; color: {color}; font-weight: 600;">
          <span>{icon}</span>
          <span>{title.upper()}</span>
        </div>
        <span style="background: #1e2433; color: #60a5fa; padding: 2px 6px; border-radius: 4px; font-size: 10px;">JOB [{job_num}/{total_jobs}]</span>
      </div>
      <div style="color: #9ca3af; font-size: 11px;">{subtitle}</div>
    </div>
    """

def parse_aria2_progress(line):
    m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)\]", line)
    if not m_size:
        m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)", line)
    if not m_size:
        return None
    downloaded, total, percent = m_size.groups()
    m_cn = re.search(r"CN:(\d+)", line)
    cn = m_cn.group(1) if m_cn else "0"
    m_dl = re.search(r"DL:([0-9.]+[KMGT]?i?B)", line)
    dl = m_dl.group(1) if m_dl else "0B"
    m_eta = re.search(r"ETA:([0-9A-Za-z]+)", line)
    eta = m_eta.group(1) if m_eta else "--"
    return {
        "downloaded": downloaded,
        "total": total,
        "percent": float(percent),
        "peers": cn,
        "speed": dl,
        "eta": eta
    }

def auto_extract_archives(download_dir):
    archive_exts = ('.rar', '.zip', '.7z', '.tar', '.gz', '.bz2', '.r00')
    extracted = False
    for root, dirs, files in os.walk(download_dir):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in archive_exts:
                if '.part' in f.lower() and not any(k in f.lower() for k in ('.part01.', '.part1.', '.part001.')):
                    continue
                archive_path = os.path.join(root, f)
                res = subprocess.run(['7z', 'x', archive_path, f'-o{root}', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                if res.returncode == 0:
                    extracted = True
                    try:
                        os.remove(archive_path)
                    except Exception:
                        pass
    return extracted

def fast_copy_file(src, dst, chunk_size=16 * 1024 * 1024):
    with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break
            fdst.write(chunk)

def send_webhook(webhook_url, title, size_str, elapsed_str, avg_speed, path_str):
    if not webhook_url or not webhook_url.startswith("http"):
        return False
    payload = {
        "embeds": [{
            "title": "✅ Download Complete & Synced",
            "description": f"**{title}** has been successfully transferred to Google Drive.",
            "color": 0x10b981,
            "fields": [
                {"name": "Verified Size", "value": size_str, "inline": True},
                {"name": "Time Elapsed", "value": elapsed_str, "inline": True},
                {"name": "Avg Throughput", "value": avg_speed, "inline": True},
                {"name": "Destination", "value": f"`{path_str}`", "inline": False}
            ],
            "footer": {"text": "Colab Cloud Download Station"}
        }]
    }
    try:
        req = urllib.request.Request(
            webhook_url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json", "User-Agent": "ColabDownloader"}
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            return resp.status in (200, 204)
    except Exception:
        return False

# 8. Execute Queue
if queue:
    print(f"\nStarting batch execution ({len(queue)} jobs queued)...")
    
    for idx, item in enumerate(queue):
        job_num = idx + 1
        total_jobs = len(queue)
        job_cache = os.path.join(BASE_CACHE_DIR, f"job_{job_num}")
        os.makedirs(job_cache, exist_ok=True)
        
        start_time = time.time()
        
        cmd = [
            "aria2c",
            "-c",
            "--enable-dht=true",
            "--bt-enable-lpd=true",
            "--max-connection-per-server=16",
            "--split=16",
            "--bt-max-peers=128",
            "--disk-cache=64M",
            "--file-allocation=falloc",
            "--bt-request-peer-speed-limit=200K",
            "--seed-time=0",
            "--summary-interval=1",
            f"--dir={job_cache}",
        ]
        if trackers_arg:
            cmd.append(f"--bt-tracker={trackers_arg}")
        cmd.append(item["target"])

        display_handle = display(HTML(render_progress_card(job_num, total_jobs, status="CONNECTING PEERS")), display_id=True)
        
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        buffer = ""
        while True:
            chunk = proc.stdout.read(256)
            if not chunk and proc.poll() is not None:
                break
            buffer += chunk
            while "\r" in buffer or "\n" in buffer:
                idx_r = buffer.find("\r")
                idx_n = buffer.find("\n")
                if idx_r != -1 and (idx_n == -1 or idx_r < idx_n):
                    line, buffer = buffer[:idx_r], buffer[idx_r + 1:]
                else:
                    line, buffer = buffer[:idx_n], buffer[idx_n + 1:]
                line = line.strip()
                if line:
                    data = parse_aria2_progress(line)
                    if data:
                        elapsed_str = format_clock(time.time() - start_time)
                        display_handle.update(HTML(render_progress_card(
                            job_num,
                            total_jobs,
                            percent=data["percent"],
                            speed=data["speed"],
                            downloaded=data["downloaded"],
                            total=data["total"],
                            elapsed_str=elapsed_str,
                            eta=data["eta"],
                            peers=data["peers"],
                            status="DOWNLOADING"
                        )))
                    elif "downloading metadata" in line.lower():
                        elapsed_str = format_clock(time.time() - start_time)
                        display_handle.update(HTML(render_progress_card(job_num, total_jobs, elapsed_str=elapsed_str, status="METADATA")))

        if proc.returncode == 0:
            # Check for compressed archives and unpack
            display_handle.update(HTML(render_status_card(job_num, total_jobs, "EXTRACTING ARCHIVES", "Checking and unpacking .rar / .zip / .7z archives...", color="#38bdf8", icon="📦")))
            extracted = auto_extract_archives(job_cache)
            
            # Stream into Google Drive via 16 MB buffered copy
            display_handle.update(HTML(render_status_card(job_num, total_jobs, "SYNCING TO DRIVE", "Streaming verified chunks sequentially (16 MB buffer) & flushing...", color="#f59e0b", icon="⟳")))
            
            total_bytes = 0
            for it in os.listdir(job_cache):
                src = os.path.join(job_cache, it)
                dst = os.path.join(DRIVE_SAVE_DIR, it)
                if os.path.isdir(src):
                    os.makedirs(dst, exist_ok=True)
                    for root, dirs, files_list in os.walk(src):
                        rel = os.path.relpath(root, src)
                        target_root = os.path.join(dst, rel)
                        os.makedirs(target_root, exist_ok=True)
                        for f in files_list:
                            s_file = os.path.join(root, f)
                            d_file = os.path.join(target_root, f)
                            fast_copy_file(s_file, d_file)
                            total_bytes += os.path.getsize(s_file)
                else:
                    fast_copy_file(src, dst)
                    total_bytes += os.path.getsize(src)

            os.sync()
            
            total_time = max(time.time() - start_time, 1)
            total_time_str = format_seconds(total_time)
            
            size_mb = total_bytes / (1024 * 1024)
            size_str = f"{size_mb / 1024:.2f} GB" if size_mb >= 1024 else f"{size_mb:.1f} MB"
            
            avg_speed_val = size_mb / total_time
            avg_speed_str = f"{avg_speed_val:.1f} MB/s"
            
            # Send Webhook alert if configured
            if webhook_url:
                send_webhook(webhook_url, item["name"], size_str, total_time_str, avg_speed_str, DRIVE_SAVE_DIR)
            
            display_handle.update(HTML(render_completed_card(job_num, total_jobs, item["name"], size_str, total_time_str, avg_speed_str, DRIVE_SAVE_DIR)))
            
            # Cleanup local job cache
            shutil.rmtree(job_cache, ignore_errors=True)
        else:
            print(f"Job {job_num} exited with error code:", proc.returncode)

    print("\n" + "="*50)
    print("🎉 ALL QUEUED JOBS FINISHED AND SYNCED TO GOOGLE DRIVE!")
    print(f"Destination: {DRIVE_SAVE_DIR}")
    print("="*50)
else:
    print("Queue is empty. No downloads initiated.")

